In [2]:
import pandas as pd
df=pd.read_csv('df_merged.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20926 entries, 0 to 20925
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            20926 non-null  int64  
 1   player_name   20925 non-null  object 
 2   games         20925 non-null  float64
 3   time          20925 non-null  float64
 4   goals         20925 non-null  float64
 5   xG            20925 non-null  float64
 6   assists       20925 non-null  float64
 7   xA            20925 non-null  float64
 8   shots         20925 non-null  float64
 9   key_passes    20925 non-null  float64
 10  yellow_cards  20925 non-null  float64
 11  red_cards     20925 non-null  float64
 12  position      20925 non-null  object 
 13  team_title    20895 non-null  object 
 14  npg           20895 non-null  float64
 15  npxG          20895 non-null  float64
 16  xGChain       20895 non-null  float64
 17  xGBuildup     20895 non-null  float64
 18  league        20895 non-nu

In [6]:
pip install fairlearn

  Using cached fairlearn-0.13.0-py3-none-any.whl.metadata (7.3 kB)
Using cached fairlearn-0.13.0-py3-none-any.whl (251 kB)
   ---------------------------------------- 0.0/430.8 kB ? eta -:--:--
    --------------------------------------- 10.2/430.8 kB ? eta -:--:--
   -- ------------------------------------ 30.7/430.8 kB 660.6 kB/s eta 0:00:01
   ---------- ----------------------------- 112.6/430.8 kB 1.3 MB/s eta 0:00:01
   ------------------- -------------------- 204.8/430.8 kB 1.6 MB/s eta 0:00:01
   ------------------------------ --------- 327.7/430.8 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------  430.1/430.8 kB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 430.8/430.8 kB 1.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference



In [ ]:
# Sample data
np.random.seed(42)
n_samples = 1000

data = {
    'experience_years': np.random.normal(5, 2, n_samples),
    'education_score': np.random.normal(75, 15, n_samples),
    'interview_score': np.random.normal(80, 10, n_samples),
    'gender': np.random.choice(['M', 'F'], n_samples),
    'age_group': np.random.choice(['young', 'old'], n_samples)
}

In [54]:
type(data)

dict

In [55]:
df = pd.DataFrame(data)

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experience_years  1000 non-null   float64
 1   education_score   1000 non-null   float64
 2   interview_score   1000 non-null   float64
 3   gender            1000 non-null   object 
 4   age_group         1000 non-null   object 
dtypes: float64(3), object(2)
memory usage: 39.2+ KB


In [57]:
df.head()

,experience_years,education_score,interview_score,gender,age_group
0,5.993428,95.990332,73.248217,F,old
1,4.723471,88.869505,78.554813,F,old
2,6.295377,75.894456,72.075801,F,young
3,8.046060,65.295948,76.920385,F,young
4,4.531693,85.473350,61.063853,M,old


In [58]:
#BIAS bolgan target qoshish
df['hired'] = (
    (df['experience_years'] > 3) & 
    (df['education_score'] > 70) & 
    (df['interview_score'] > 75) &
    # Bias: erkak va yoshlarga tarafkashlik qilinishi
    (np.random.random(n_samples) < 
     np.where(df['gender'] == 'M', 0.8, 0.6) * 
     np.where(df['age_group'] == 'young', 0.9, 0.7))
).astype(int)   

In [59]:
import numpy as np
# np.where(condition, value_if_true, value_if_false)

ages = np.array([18, 25, 16, 40])

result = np.where(ages >= 25, "Adult", "Minor")
print(result)


['Minor' 'Adult' 'Minor' 'Adult']


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   experience_years  1000 non-null   float64
 1   education_score   1000 non-null   float64
 2   interview_score   1000 non-null   float64
 3   gender            1000 non-null   object 
 4   age_group         1000 non-null   object 
 5   hired             1000 non-null   int32  
dtypes: float64(3), int32(1), object(2)
memory usage: 43.1+ KB


In [61]:
df.head()

,experience_years,education_score,interview_score,gender,age_group,hired
0,5.993428,95.990332,73.248217,F,old,0
1,4.723471,88.869505,78.554813,F,old,1
2,6.295377,75.894456,72.075801,F,young,0
3,8.046060,65.295948,76.920385,F,young,0
4,4.531693,85.473350,61.063853,M,old,0


In [62]:
# Featurelar
features = ['experience_years', 'education_score', 'interview_score']
X = df[features]
y = df['hired']
sensitive_features = df[['gender', 'age_group']]

In [63]:
# Split data
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y, sensitive_features, test_size=0.3, random_state=42
)

In [64]:
# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [65]:
# predictions
y_pred = model.predict(X_test)

In [66]:
# Calculate fairness metrics
print("Fairness Metrics Analysis")
print("=" * 30)


Fairness Metrics Analysis


In [67]:
# Demographic Parity(gender)
dp_gender = demographic_parity_difference(
    y_test, y_pred, sensitive_features=sf_test['gender']
)
print(f"Demographic Parity Difference (Gender): {dp_gender:.3f}")

Demographic Parity Difference (Gender): 0.017


In [68]:
# Demographic Parity()
dp_age = demographic_parity_difference(
    y_test, y_pred, sensitive_features=sf_test['age_group']
)
print(f"Demographic Parity Difference (Age): {dp_age:.3f}")

Demographic Parity Difference (Age): 0.003


In [69]:
# Equalized Odds(True positive va false positive)
eo_gender = equalized_odds_difference(
    y_test, y_pred, sensitive_features=sf_test['gender']
)
print(f"Equalized Odds Difference (Gender): {eo_gender:.3f}")

Equalized Odds Difference (Gender): 0.039


In [ ]:
# Equalized Odds(True positive va false positive)
eo_gender = equalized_odds_difference(
    y_test, y_pred, sensitive_features=sf_test['age_group']
)
print(f"Equalized Odds Difference (Gender): {eo_gender:.3f}")

In [70]:
# Interpretation helper(modelni tushunish)
def interpret_fairness_score(score, metric_name):
    if abs(score) < 0.05:
        return f"✅ {metric_name}: Excellent fairness (difference: {score:.3f})"
    elif abs(score) < 0.10:
        return f"⚠️  {metric_name}: Acceptable fairness (difference: {score:.3f})"
    else:
        return f"❌ {metric_name}: Poor fairness (difference: {score:.3f})"

print("\nInterpretation:")
print(interpret_fairness_score(dp_gender, "Demographic Parity (Gender)"))
print(interpret_fairness_score(eo_gender, "Equalized Odds (Gender)"))


Interpretation:
✅ Demographic Parity (Gender): Excellent fairness (difference: 0.017)
✅ Equalized Odds (Gender): Excellent fairness (difference: 0.039)


# Solution 1 : Reweighting (Bias Mitigation)

In [71]:
#Reweighting

In [72]:
train_gender = sf_train['gender'].values
weights = np.where(train_gender == 'F', 1.3, 1.0) 

In [73]:

model_rw = RandomForestClassifier(random_state=42)
model_rw.fit(X_train, y_train, sample_weight=weights)

RandomForestClassifier(random_state=42)

In [74]:
y_pred_rw = model_rw.predict(X_test)

In [75]:
dp_gender_rw = demographic_parity_difference(y_test, y_pred_rw, sensitive_features=sf_test['gender'])
eo_gender_rw = equalized_odds_difference(y_test, y_pred_rw, sensitive_features=sf_test['gender'])

print("\n=== After Reweighing (Gender) ===")
print(f"Demographic Parity Difference (Gender): {dp_gender_rw:.3f}")
print(f"Equalized Odds Difference (Gender): {eo_gender_rw:.3f}")


=== After Reweighing (Gender) ===
Demographic Parity Difference (Gender): 0.005
Equalized Odds Difference (Gender): 0.121


# Solution 2: ThresholdOptimizer(Bias Mitigation)

In [76]:
from fairlearn.postprocessing import ThresholdOptimizer

# Post-process predictions for fairness
post_processor = ThresholdOptimizer(
    estimator=model,
    constraints='demographic_parity',
    prefit=True
)

post_processor.fit(X_train, y_train, sensitive_features=sf_train['gender'])
fair_post_predictions = post_processor.predict(X_test, sensitive_features=sf_test['gender'])

print("Post-processed model fairness:")
print(f"Demographic Parity: {demographic_parity_difference(y_test, fair_post_predictions, sensitive_features=sf_test['gender']):.3f}")

Post-processed model fairness:
Demographic Parity: 0.101


In [77]:
print(f"Equalized Odds: {equalized_odds_difference(y_test, fair_post_predictions, sensitive_features=sf_test['gender']):.3f}")


Equalized Odds: 0.155


In [79]:
from fairlearn.postprocessing import ThresholdOptimizer

# Post-process predictions for fairness
post_processor = ThresholdOptimizer(
    estimator=model,
    constraints='demographic_parity',
    prefit=True
)

post_processor.fit(X_train, y_train, sensitive_features=sf_train['age_group'])
fair_post_predictions = post_processor.predict(X_test, sensitive_features=sf_test['age_group'])

print("Post-processed model fairness:")
print(f"Demographic Parity: {demographic_parity_difference(y_test, fair_post_predictions, sensitive_features=sf_test['age_group']):.3f}")

Post-processed model fairness:
Demographic Parity: 0.139


In [80]:
print(f"Equalized Odds: {equalized_odds_difference(y_test, fair_post_predictions, sensitive_features=sf_test['age_group']):.3f}")


Equalized Odds: 0.301
